# The Price Is Right

### And now, to evaluate our fine-tuned open source model



## Overview

This notebook evaluates a **fine-tuned Llama 3.2 3B** model on the task of predicting product prices from text descriptions. The model was fine-tuned using QLoRA in the companion `training-full.ipynb` notebook, and its adapter weights were pushed to the Hugging Face Hub.

**What this notebook does:**
1. Loads the quantized base model and applies the fine-tuned LoRA adapter from the Hub
2. Defines a prediction function that generates price estimates from product descriptions
3. Runs the `Tester` evaluation harness (from `util.py`) on the held-out test set
4. Visualizes prediction accuracy with scatter plots, error distributions, and running error trends

**Baselines for comparison:**
- Human-level performance: ~$87.62 average error
- GPT-4.1-nano: ~$62.51 average error

## 1. Environment Setup

Install dependencies and fetch the evaluation utility module. The `util.py` helper provides the `Tester` class and `evaluate()` function, which handle prediction, error calculation, and visualization.

In [ ]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

### Import Libraries

Key additions compared to the training notebook:
- **`PeftModel`** — loads the LoRA adapter weights on top of the quantized base model
- **`evaluate`** from `util.py` — orchestrates the test loop: runs predictions, computes errors, and generates charts
- **`plotly.io`** — configured for inline notebook rendering of interactive charts

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
# from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate
import plotly.io as pio
from getpass import getpass


pio.renderers.default = "notebook_connected"

## 2. Configuration

Set up the model identifiers and point to the correct fine-tuned checkpoint on the Hub.

- **`HUB_MODEL_NAME`** — the repository path on Hugging Face Hub where the LoRA adapter was saved during training (e.g., `patrickcmd/price-2026-03-05_20.25.01`).
- **`REVISION`** — an optional git commit hash pinning a specific checkpoint. This is useful when training was interrupted and multiple checkpoints exist; omit it (set to `None`) to load the latest version.
- **`QUANT_4_BIT`** — must match the quantization used during training so that the adapter weights align with the base model's weight format.

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "patrickcmd" # your HF name here!

LITE_MODE = False  # True or False

DATA_USER = "patrickcmd"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2026-03-05_18.21.18-lite"
  REVISION = None
else:
  RUN_NAME = "2026-03-05_20.25.01"
  REVISION = "d9cb3f8b8262bada25fb00ae1b405e69ae9607e7"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

## 3. Authentication

### Log in to HuggingFace

A HuggingFace token with **read access** is needed to download the fine-tuned adapter weights from your private repository. Sign up at [huggingface.co](https://huggingface.co), create a token, and either add it as a Colab secret named `HF_TOKEN` or enter it interactively below.

In [ ]:
# Log in to HuggingFace

# hf_token = userdata.get('HF_TOKEN')
# login(hf_token, add_to_git_credential=True)
login()

## 4. Load the Test Dataset

We load the same dataset used during training and extract only the **test split** — a held-out set of 10,000 examples the model has never seen. Each example contains a `prompt` (the product description ending with "Price is $") and a `completion` (the ground-truth price).

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

Inspect a sample to see the prompt format — the model must generate a price value immediately after the trailing `$` symbol.

In [ ]:
test[0]

## 5. Load the Tokenizer and Fine-Tuned Model

The loading process has two stages:

1. **Load the base model** — `Llama-3.2-3B` is loaded in 4-bit quantized form, identical to how it was loaded during training. This ensures the weight dimensions match the trained LoRA adapters.
2. **Apply the LoRA adapter** — `PeftModel.from_pretrained()` downloads the adapter weights from the Hub and layers them on top of the frozen base model. If a specific `REVISION` is set, it loads that exact checkpoint; otherwise, it uses the latest commit.

The combined model (base + adapter) will be larger than the base alone because the LoRA matrices are loaded in full precision alongside the 4-bit base weights.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

Inspect the model architecture to verify that LoRA adapters (`lora_A`, `lora_B`) are attached to the expected layers. Each targeted linear layer should show `lora.Linear4bit` with the configured rank (256 in full mode).

In [ ]:
fine_tuned_model

## 6. Evaluate the Fine-Tuned Model

### Inference Pipeline

The `model_predict` function takes a single test example, tokenizes its prompt, and generates up to 8 new tokens. Only the generated tokens (after the prompt) are decoded — this is the model's price prediction.

**How `evaluate()` works** (from `util.py`):
- Runs `model_predict` on 200 test examples (default sample size)
- Extracts the numeric value from the generated text using regex
- Computes absolute error (`|predicted - actual|`) for each example
- Color-codes results: green (< $40 or < 20% error), orange (< $80 or < 40%), red (worse)
- Produces a scatter plot (predicted vs. actual) and a running error trend chart with 95% CI

### Baselines

| Model | Avg Error |
|-------|-----------|
| Human estimate | ~$87.62 |
| GPT-4.1-nano | ~$62.51 |
| **Our fine-tuned Llama 3.2 3B** | *(see results below)* |

**Caveat:** Product prices vary considerably. The model cannot account for sale prices, regional pricing, or other factors absent from the product description.

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

Run the evaluation. We set a fixed seed for reproducibility so results are consistent across runs. The `evaluate()` function prints color-coded errors in real-time, then displays the final charts.

In [ ]:
set_seed(42)
evaluate(model_predict, test)

## Interpreting the Results

The evaluation produces two visualizations:

1. **Error Trend Chart** — Shows the running average error with a 95% confidence interval as more test examples are processed. A stable, converging line indicates consistent model performance.
2. **Scatter Plot (Predicted vs. Actual)** — Points near the dashed y=x line are accurate predictions. Color coding highlights the error magnitude: green (good), orange (moderate), red (poor).

**Key metrics reported:**
- **Average Error** — Mean absolute difference between predicted and actual price
- **MSE** — Mean Squared Error, penalizes large misses more heavily
- **r²** — Coefficient of determination; 100% means perfect predictions, 0% means no better than predicting the mean

Compare these results against the baselines in Section 6 to assess how well the fine-tuned 3B model performs relative to larger commercial models and human estimates.